* Prompting strategies
* Data filtering and curation
* Synthetic data generation
* Reinforcement learning
* Lightweight fine-tuning
* Or other approaches of your choice

evaluation model: https://www.kaggle.com/models/metric/nemotron-3-nano-30b-a3b-bf16

## Discussion
- https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge/discussion/681745
- https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge/discussion/681745#3453015

## Fine Tune
### 🦥 Unsloth
- 2× faster training, 80% less VRAM, zero accuracy loss
-  Colab-friendly, works with ~3GB VRAM
- Supports Qwen 3, Llama 4, Gemma 4, Mistral

### 🏭 LLaMA-Factory
- 100+ models, WebUI or CLI, no coding required
- Full fine-tuning, LoRA, QLoRA, DPO all in one
- 49k GitHub stars and growing

### ⚡ DeepSpeed
- Multi-GPU, multi-node, ZeRO memory optimization
- Fine-tune models with trillions of parameters
- Production-tested at the world's top AI labs

### 🐙 Axolotl
- YAML config → fine-tuned model, no boilerplate
- LoRA, QLoRA, DPO, GRPO, multimodal support
- Most streamlined DX of any framework

### 🤗 TRL (v1.0 just dropped, March 2026)
- The library behind DeepSeek-R1's GRPO training
- Full post-training stack: SFT, DPO, GRPO, PPO, reward modeling
- 18k stars, actively maintained by Hugging Face

### 🔧 PEFT
- The LoRA/QLoRA engine that powers everything else
- Fine-tune 12B+ models on consumer hardware
- Checkpoints in MBs, not GBs, deploy anywhere


### Links
* 🦥 Unsloth → github.com/unslothai/unsloth
* 🏭 LLaMA-Factory → github.com/hiyouga/LLaMA-Factory
* ⚡ DeepSpeed → github.com/microsoft/DeepSpeed
* 🐙 Axolotl → github.com/axolotl-ai-cloud/axolotl
* 🤗 TRL v1.0 → github.com/huggingface/trl
* 🔧 PEFT → github.com/huggingface/peft

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/models/huikang/nemotron-adapter/transformers/default/27/adapter_model.safetensors
/kaggle/input/models/huikang/nemotron-adapter/transformers/default/27/adapter_config.json
/kaggle/input/models/huikang/nemotron-adapter/transformers/default/27/README.md
/kaggle/input/models/huikang/nemotron-adapter/transformers/default/27/checkpoint_complete
/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv
/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/test.csv


In [3]:
import pandas as pd
import numpy as np

## Data check

In [4]:
train  = pd.read_csv("/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/test.csv")

In [5]:
train.head()

,id,prompt,answer
0,00066667,"In Alice's Wonderland, a secret bit manipulati...",10010111
1,000b53cf,"In Alice's Wonderland, a secret bit manipulati...",01000011
2,00189f6a,"In Alice's Wonderland, secret encryption rules...",cat imagines book
3,001b24c4,"In Alice's Wonderland, numbers are secretly co...",XXXVIII
4,001c63cb,"In Alice's Wonderland, secret encryption rules...",wizard creates secret


In [6]:
# .iloc[0] returns a Series; .items() gives you (column_name, value) pairs
for column, value in train.iloc[0].items():
    print("=============" * 4)
    print(column,"\n")
    print(value)

id 

00066667
prompt 

In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.

Here are some examples of input -> output:
01010001 -> 11011101
00001001 -> 01101101
00010101 -> 01010101
11111111 -> 10000001
10011101 -> 01000101
00111011 -> 00001001
10111101 -> 00000101
00100110 -> 10110011

Now, determine the output for: 00110100
answer 

10010111


In [7]:
for column, value in train.iloc[-1].items():
    print("=============" * 4)
    print(column,"\n")
    print(value)

id 

ffe680c2
prompt 

In Alice's Wonderland, the gravitational constant has been secretly changed. Here are some example observations:
For t = 3.16s, distance = 38.4 m
For t = 3.55s, distance = 48.46 m
For t = 4.6s, distance = 81.37 m
For t = 3.16s, distance = 38.4 m
For t = 1.51s, distance = 8.77 m
Now, determine the falling distance for t = 3.85s given d = 0.5*g*t^2.
answer 

57.0


In [8]:
"KAGGLE_KERNEL_RUN_TYPE" in os.environ

True

In [7]:
!uv pip install --no-cache-dir --force-reinstall "tinker-cookbook @ git+https://github.com/thinking-machines-lab/tinker-cookbook.git@nightly"

Using Python 3.12.12 environment at: /usr
Resolved 93 packages in 3.44s                                        
Prepared 93 packages in 20.41s                                           
Uninstalled 75 packages in 2.89s
Installed 93 packages in 388ms                              
 ~ aiohappyeyeballs==2.6.1
 - aiohttp==3.13.3
 + aiohttp==3.13.5
 ~ aiosignal==1.4.0
 ~ annotated-doc==0.0.4
 ~ annotated-types==0.7.0
 - anyio==4.12.1
 + anyio==4.13.0
 - attrs==25.4.0
 + attrs==26.1.0
 ~ blobfile==3.2.0
 - certifi==2026.1.4
 + certifi==2026.4.22
 - charset-normalizer==3.4.4
 + charset-normalizer==3.4.7
 + chz==0.4.0
 - click==8.3.1
 + click==8.3.3
 ~ cloudpickle==3.1.2
 - cuda-bindings==12.9.4
 + cuda-bindings==13.2.0
 - cuda-pathfinder==1.3.5
 + cuda-pathfinder==1.5.4
 - cuda-toolkit==12.8.1
 + cuda-toolkit==13.0.2
 - datasets==4.8.3
 + datasets==4.8.5
 ~ dill==0.4.1
 ~ distro==1.9.0
 - filelock==3.24.3
 + filelock==3.29.0
 ~ frozenlist==1.8.0
 ~ fsspec==2026.2.0
 ~ h11==0.16.0
 ~ h2==4.3.0


In [8]:
import torch
import shutil
import tinker_cookbook.weights._adapter as A
from tinker_cookbook import weights

### LoRA Adapter Fusion and Rank Compression Pipeline



In [9]:
FORCED_FUSED_RANK = 32


def _compress_lora_pair_to_rank(B: torch.Tensor, A_mat: torch.Tensor, rank: int):
    # Compute delta = B @ A
    delta = B.float() @ A_mat.float()

    # Perform SVD decomposition
    U, S, Vh = torch.linalg.svd(delta, full_matrices=False)

    # Truncate to desired rank
    U = U[:, :rank]
    S = S[:rank]
    Vh = Vh[:rank, :]

    # Reconstruct LoRA matrices
    sroot = torch.sqrt(S)
    B_new = U * sroot.unsqueeze(0)
    A_new = sroot.unsqueeze(1) * Vh

    return B_new.to(B.dtype).contiguous(), A_new.to(A_mat.dtype).contiguous()



def patched_merge_fused_projections(
    fused_model_key: str,
    adapter_layer_prefix: str,
    components,
    model_state_shapes,
    peft_weights,
    target_modules,
    profile,
) -> int:

    fused_out_dim = model_state_shapes[fused_model_key][0]
    fused_target_name = fused_model_key.removesuffix(".weight").rsplit(".", 1)[-1]

    # Find component order
    component_order = None
    for target, comps in profile.fused_projection_map:
        if target == fused_target_name:
            component_order = comps
            break

    assert component_order is not None, "Component order not found"

    comp_by_name = {name: (lora_A, lora_B) for name, lora_A, lora_B in components}

    lora_A_parts = []
    comp_slices = []
    merged_rank = 0
    row_offset = 0

    # Merge components
    for comp_name in component_order:
        if comp_name not in comp_by_name:
            raise RuntimeError(f"Missing component {comp_name}")

        lora_A, lora_B = comp_by_name[comp_name]

        r = lora_A.shape[0]
        out_dim = lora_B.shape[0]

        lora_A_parts.append(lora_A)
        comp_slices.append((row_offset, row_offset + out_dim, r))

        row_offset += out_dim
        merged_rank += r

    merged_lora_A = torch.cat(lora_A_parts, dim=0)

    merged_lora_B = torch.zeros(
        fused_out_dim,
        merged_rank,
        dtype=merged_lora_A.dtype,
        device=merged_lora_A.device,
    )

    # Fill B matrix
    rank_offset = 0
    for i, (row_start, row_end, r) in enumerate(comp_slices):
        _, lora_B = comp_by_name[component_order[i]]
        merged_lora_B[row_start:row_end, rank_offset:rank_offset + r] = lora_B
        rank_offset += r

    # Compress to rank 32 if needed
    final_rank = merged_rank
    if merged_rank > FORCED_FUSED_RANK:
        merged_lora_B, merged_lora_A = _compress_lora_pair_to_rank(
            merged_lora_B,
            merged_lora_A,
            FORCED_FUSED_RANK
        )
        final_rank = FORCED_FUSED_RANK

    # Save result
    peft_target_key = f"{adapter_layer_prefix}.{fused_target_name}.weight"
    A._add_peft_weight(
        peft_target_key,
        merged_lora_A,
        merged_lora_B,
        peft_weights,
        target_modules
    )

    return final_rank


A._merge_fused_projections = patched_merge_fused_projections
print("Patched function:", A._merge_fused_projections.__name__)


weights.build_lora_adapter(
    base_model="nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16",
    adapter_path="/kaggle/input/models/huikang/nemotron-adapter/transformers/default/27",
    output_path="/kaggle/working/nemotron-adapter-ready-to-submit",
)



shutil.make_archive(
    '/kaggle/working/submission',
    'zip',
    '/kaggle/working/nemotron-adapter-ready-to-submit'
)

print("ZIP archive successfully created!")

Patched function: patched_merge_fused_projections


Fetching 35 files:   0%|          | 0/35 [00:00<?, ?it/s]

MoE expert LoRA serving for nemotron models is experimental in vLLM and not yet supported in SGLang. The adapter will be produced but may not work with all serving configurations.


ZIP archive successfully created!


In [10]:
# import tinker_cookbook
# from tinker_cookbook import weights
# import importlib.metadata as md

# print("tinker_cookbook path:", tinker_cookbook.__file__)
# print("version:", md.version("tinker-cookbook"))
# print("has build_lora_adapter:", hasattr(weights, "build_lora_adapter"))

In [13]:
!kaggle competitions submit -c nvidia-nemotron-model-reasoning-challenge -f /kaggle/working/submission.zip -k uomml23msc239362l/NVIDIA_Nemo_tron_reasoning_ch1 -v 0 -m "submission1"

403 Client Error: Forbidden for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/CreateCodeSubmission


## TRL

In [1]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 17.9 MB/s eta 0:00:0000:01


In [9]:
from trl import SFTTrainer
from datasets import load_dataset

dataset = load_dataset("trl-lib/Capybara", split="train")

# trainer = SFTTrainer(
#     model="Qwen/Qwen2.5-0.5B",
#     train_dataset=dataset,
# )
# trainer.train()

README.md:   0%|          | 0.00/520 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/37.2M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/482k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15806 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/200 [00:00<?, ? examples/s]